# ProteinGym quickstart

Download and standardize one fixed ProteinGym substitutions assay through the public API. The first run retrieves shared ProteinGym resources; later runs reuse the filesystem cache.

In [ ]:
from pathlib import Path

from dms_parser import (
    FilesystemCache,
    download_and_standardize_dataset,
    get_dataset_metadata,
    read_table,
)

In [ ]:
DATASET_ID = "BLAT_ECOLX_Jacquier_2013"
CACHE_DIR = Path("datasets/.cache")
OUTPUT_DIR = Path("datasets/proteingym/BLAT_ECOLX_Jacquier_2013")

cache = FilesystemCache(CACHE_DIR)

## Inspect normalized metadata

`get_dataset_metadata` presents the source catalog through a small, shared record shape.

In [ ]:
metadata = get_dataset_metadata(
    "proteingym",
    DATASET_ID,
    variant_type="substitutions",
    cache=cache,
)

{
    "source": metadata.source,
    "dataset_id": metadata.dataset_id,
    "title": metadata.title,
    "target_id": metadata.target_id,
    "variant_type": metadata.variant_type,
    "n_variants": metadata.n_variants,
}

## Download and standardize

The source score is retained as `score_raw`; no optional transformations are requested here. `overwrite=True` makes the example safe to rerun.

In [ ]:
result = download_and_standardize_dataset(
    "proteingym",
    DATASET_ID,
    output_dir=OUTPUT_DIR,
    cache=cache,
    overwrite=True,
)

## Inspect the standardized result

In [ ]:
dataset = read_table(result.dataset_path)
preview_columns = [
    column
    for column in (
        "variant",
        "is_wildtype",
        "n_mutations",
        "score_raw",
        "status",
        "error",
    )
    if column in dataset.columns
]
dataset.loc[:, preview_columns].head()

In [ ]:
dataset["status"].value_counts(dropna=False).rename_axis("status").to_frame("rows")

In [ ]:
{
    "dataset_path": result.dataset_path,
    "summary_csv_path": result.summary_csv_path,
    "summary_json_path": result.summary_json_path,
    "summary": {
        key: result.summary.get(key)
        for key in (
            "status",
            "raw_rows",
            "output_rows",
            "validated_rows",
            "wt_sequence_provenance",
        )
    },
}